# Bed monitoring: agentic vision pipeline

Video → YOLO pose → features → rules → smoothing → timeline → bed events → VLM agent → state machine → alerts → evaluation → outputs.

To analyze another video, change `VIDEO` in the setup cell and run all cells from the top.

## 0. Setup

In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from ultralytics import YOLO

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VIDEO = ROOT / "data/videos/IMG_5673.MOV"
SAMPLE_FPS = 2
MAX_WIDTH = 960

load_dotenv(ROOT / ".env")
VLM_MODEL = os.getenv("VLM_MODEL", "")

OUT = ROOT / "outputs" / "notebook" / VIDEO.stem
OUT.mkdir(parents=True, exist_ok=True)

print("video exists:", VIDEO.exists(), "|", VIDEO)
print("outputs:", OUT)
print("VLM model:", VLM_MODEL or "not set")

## 1. Load and sample the video

In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ok, frame = cap.read()
cap.release()

print(f"fps: {fps:.2f} | frames: {n_frames} | duration: {n_frames / fps:.1f}s | size: {width}x{height}")

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f"First frame {frame.shape}")
plt.axis("off")
plt.show()

In [ ]:
step = max(1, round(fps / SAMPLE_FPS))
samples = []

cap = cv2.VideoCapture(str(VIDEO))
idx = 0
while cap.grab():
    if idx % step == 0:
        ok, frame = cap.retrieve()
        if not ok:
            break
        scale = min(1.0, MAX_WIDTH / frame.shape[1])
        frame = cv2.resize(frame, None, fx=scale, fy=scale)
        samples.append({"frame_idx": idx, "t": idx / fps, "image": frame})
    idx += 1
cap.release()

print(f"Kept every {step}th frame -> {len(samples)} samples at ~{SAMPLE_FPS} fps")

In [ ]:
def show_grid(items, cols=4, limit=16):
    items = items[:limit]
    rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, s in zip(axes.flat, items):
        ax.imshow(cv2.cvtColor(s["image"], cv2.COLOR_BGR2RGB))
        ax.set_title(f"t={s['t']:.1f}s (frame {s['frame_idx']})", fontsize=9)
    plt.tight_layout()
    plt.show()

show_grid(samples[::max(1, len(samples) // 16)])

## 2. YOLO pose

In [ ]:
model = YOLO("yolo11n-pose.pt")

s = samples[min(40, len(samples) - 1)]
r = model(s["image"], verbose=False)[0]

print("people:", len(r.boxes))
print("box confidences:", r.boxes.conf.tolist())
print("keypoints shape:", tuple(r.keypoints.xy.shape))

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
plt.title(f"t={s['t']:.1f}s")
plt.axis("off")
plt.show()

In [ ]:
for s in samples:
    s["result"] = model(s["image"], verbose=False)[0]

for s in samples[:20]:
    r = s["result"]
    n = len(r.boxes)
    conf = f"{r.boxes.conf.max():.2f}" if n else "-"
    print(f"t={s['t']:6.1f}s | people: {n} | best conf: {conf}")

In [ ]:
annotated = [{"t": s["t"], "frame_idx": s["frame_idx"], "image": s["result"].plot()} for s in samples]
show_grid(annotated[::max(1, len(annotated) // 16)])

## 3. Features

In [ ]:
KP_NAMES = ["nose", "l_eye", "r_eye", "l_ear", "r_ear",
            "l_shoulder", "r_shoulder", "l_elbow", "r_elbow", "l_wrist", "r_wrist",
            "l_hip", "r_hip", "l_knee", "r_knee", "l_ankle", "r_ankle"]

s = next((x for x in samples if len(x["result"].boxes)), samples[0])
r = s["result"]
best = int(r.boxes.conf.argmax())
xy = r.keypoints.xy[best].numpy()
conf = r.keypoints.conf[best].numpy()

pd.DataFrame({"name": KP_NAMES, "x": xy[:, 0].round(1), "y": xy[:, 1].round(1), "conf": conf.round(2)})

In [ ]:
KP_TH = 0.5

def kp(xy, conf, i):
    return xy[i] if conf[i] >= KP_TH else None

def midpoint(xy, conf, a, b):
    pa, pb = kp(xy, conf, a), kp(xy, conf, b)
    if pa is not None and pb is not None:
        return (pa + pb) / 2
    return pa if pa is not None else pb

def torso_angle(shoulders, hips):
    dx, dy = hips - shoulders
    return np.degrees(np.arctan2(abs(dy), abs(dx)))

def joint_angle(a, b, c):
    v1, v2 = a - b, c - b
    cos = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

shoulders = midpoint(xy, conf, 5, 6)
hips = midpoint(xy, conf, 11, 12)
print("shoulders:", shoulders, "| hips:", hips)
if shoulders is not None and hips is not None:
    print(f"torso angle: {torso_angle(shoulders, hips):.1f}°  (0 = horizontal/lying, 90 = vertical/upright)")

h, k, a = kp(xy, conf, 11), kp(xy, conf, 13), kp(xy, conf, 15)
if h is not None and k is not None and a is not None:
    print(f"left knee angle: {joint_angle(h, k, a):.1f}°  (180 = straight, ~90 = bent)")

In [ ]:
img = cv2.cvtColor(s["image"], cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 6))
plt.imshow(img)
for (x, y), c in zip(xy, conf):
    plt.scatter(x, y, s=40, color="lime" if c >= KP_TH else "red")
if shoulders is not None and hips is not None:
    plt.plot([shoulders[0], hips[0]], [shoulders[1], hips[1]], color="yellow", lw=3, label="torso")
    plt.legend()
plt.title(f"t={s['t']:.1f}s | green = confident, red = ignored")
plt.axis("off")
plt.show()

In [ ]:
det = YOLO("yolo11n.pt")

bed_box = None
for s_ in samples[::max(1, len(samples) // 10)]:
    b = det(s_["image"], classes=[59], verbose=False)[0]
    if len(b.boxes):
        areas = (b.boxes.xyxy[:, 2] - b.boxes.xyxy[:, 0]) * (b.boxes.xyxy[:, 3] - b.boxes.xyxy[:, 1])
        bed_box = b.boxes.xyxy[int(areas.argmax())].numpy()
        break

if bed_box is not None:
    x1, y1, x2, y2 = bed_box
    bed_poly = np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], dtype=np.int32)
    print("bed detected:", bed_box.round(1))
else:
    bed_poly = np.array([[100, 200], [500, 200], [500, 400], [100, 400]], dtype=np.int32)
    print("No bed detected: using a manual polygon. Edit these coordinates to match the bed.")

In [ ]:
def in_bed(point, poly):
    return cv2.pointPolygonTest(poly, (float(point[0]), float(point[1])), False) >= 0

img = cv2.cvtColor(s["image"], cv2.COLOR_BGR2RGB).copy()
cv2.polylines(img, [bed_poly], True, (0, 0, 255), 2)
plt.figure(figsize=(10, 6))
plt.imshow(img)
if hips is not None:
    plt.scatter(*hips, s=120, color="yellow", edgecolors="black")
    plt.title(f"hips in bed: {in_bed(hips, bed_poly)}")
plt.axis("off")
plt.show()

In [ ]:
rows = []
for s_ in samples:
    r_ = s_["result"]
    row = {"t": s_["t"], "frame_idx": s_["frame_idx"], "people": len(r_.boxes)}
    if len(r_.boxes):
        i = int(r_.boxes.conf.argmax())
        xy_, c_ = r_.keypoints.xy[i].numpy(), r_.keypoints.conf[i].numpy()
        sh, hp = midpoint(xy_, c_, 5, 6), midpoint(xy_, c_, 11, 12)
        kh, kk, ka = kp(xy_, c_, 11), kp(xy_, c_, 13), kp(xy_, c_, 15)
        x1, y1, x2, y2 = r_.boxes.xyxy[i].numpy()
        row.update(
            box_conf=float(r_.boxes.conf[i]),
            visible_kp=int((c_ >= KP_TH).sum()),
            torso_angle=torso_angle(sh, hp) if sh is not None and hp is not None else np.nan,
            knee_angle=joint_angle(kh, kk, ka) if kh is not None and kk is not None and ka is not None else np.nan,
            hip_in_bed=in_bed(hp, bed_poly) if hp is not None else np.nan,
            hip_x=hp[0] if hp is not None else np.nan,
            hip_y=hp[1] if hp is not None else np.nan,
            box_h=y2 - y1,
        )
    rows.append(row)

df = pd.DataFrame(rows)
for col in ["box_conf", "visible_kp", "torso_angle", "knee_angle", "hip_in_bed", "hip_x", "hip_y", "box_h"]:
    if col not in df:
        df[col] = np.nan
df["visible_kp"] = df["visible_kp"].fillna(0)

dist = np.hypot(df["hip_x"].diff(), df["hip_y"].diff())
df["speed"] = (dist / df["t"].diff() / df["box_h"]).rolling(3, min_periods=1).mean()
df.head(20)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df["t"], df["torso_angle"], marker=".")
axes[0].axhline(35, color="green", ls="--", label="lying below")
axes[0].axhline(55, color="orange", ls="--", label="upright above")
axes[0].set_ylabel("torso angle (°)")
axes[0].legend(loc="upper right")

axes[1].plot(df["t"], df["knee_angle"], marker=".", color="purple")
axes[1].set_ylabel("knee angle (°)")

axes[2].step(df["t"], df["hip_in_bed"].astype(float), where="post", color="red")
axes[2].set_ylabel("hips in bed")
axes[2].set_yticks([0, 1])

axes[3].plot(df["t"], df["visible_kp"], marker=".", color="gray")
axes[3].set_ylabel("visible keypoints")
axes[3].set_xlabel("time (s)")

plt.tight_layout()
plt.show()

In [ ]:
T = 25.0
s_ = min(samples, key=lambda x: abs(x["t"] - T))
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(s_["result"].plot(), cv2.COLOR_BGR2RGB))
plt.title(f"t={s_['t']:.1f}s")
plt.axis("off")
plt.show()
df.iloc[(df["t"] - T).abs().argmin()]

## 4. Rules: one state per frame

In [ ]:
LYING_MAX, UPRIGHT_MIN = 35, 55
KNEE_BENT_MAX = 140
WALK_SPEED = 0.25
MIN_VISIBLE_KP = 4

def classify(row):
    if row["people"] == 0:
        return "NO_PERSON"
    if row["visible_kp"] < MIN_VISIBLE_KP:
        return "UNKNOWN"
    angle, bed = row["torso_angle"], row["hip_in_bed"]
    if pd.isna(angle) or pd.isna(bed) or LYING_MAX < angle < UPRIGHT_MIN:
        return "UNKNOWN"
    if angle <= LYING_MAX:
        return "LYING_IN_BED" if bed else "LYING_OUTSIDE_BED"
    if bed:
        return "SITTING_ON_BED"
    if not pd.isna(row["knee_angle"]) and row["knee_angle"] <= KNEE_BENT_MAX:
        return "SITTING_OUTSIDE_BED"
    if not pd.isna(row["speed"]) and row["speed"] >= WALK_SPEED:
        return "WALKING"
    return "STANDING"

IN_STATES = {"LYING_IN_BED", "SITTING_ON_BED"}
OUT_STATES = {"SITTING_OUTSIDE_BED", "STANDING", "WALKING", "OUT_OF_BED", "LYING_OUTSIDE_BED"}

last_known = None
raw = []
for s_state in df.apply(classify, axis=1):
    if s_state == "NO_PERSON":
        s_state = "OUT_OF_BED" if last_known in OUT_STATES else "UNKNOWN"
    elif s_state != "UNKNOWN":
        last_known = s_state
    raw.append(s_state)
df["raw_state"] = raw
df["raw_state"].value_counts()

In [ ]:
STATE_ORDER = ["LYING_IN_BED", "SITTING_ON_BED", "STANDING", "WALKING",
               "SITTING_OUTSIDE_BED", "OUT_OF_BED", "LYING_OUTSIDE_BED", "UNKNOWN"]

def plot_states(series_dict, title=""):
    fig, axes = plt.subplots(len(series_dict), 1, figsize=(14, 2.5 * len(series_dict)), sharex=True, squeeze=False)
    for ax, (name, sts) in zip(axes[:, 0], series_dict.items()):
        y = [STATE_ORDER.index(x) for x in sts]
        ax.step(df["t"], y, where="post")
        ax.scatter(df["t"], y, s=8)
        ax.set_yticks(range(len(STATE_ORDER)), STATE_ORDER, fontsize=8)
        ax.set_title(name, fontsize=10)
        ax.grid(axis="y", alpha=0.3)
    axes[-1, 0].set_xlabel("time (s)")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_states({"raw": df["raw_state"]}, "Per-frame states")

## 5. Smoothing

In [ ]:
from collections import Counter

def majority_filter(states, window):
    half = window // 2
    out = []
    for i in range(len(states)):
        counts = Counter(states[max(0, i - half): i + half + 1]).most_common()
        top = {x for x, n in counts if n == counts[0][1]}
        out.append(states[i] if states[i] in top else counts[0][0])
    return out

def segments(states):
    segs, start = [], 0
    for i in range(1, len(states) + 1):
        if i == len(states) or states[i] != states[start]:
            segs.append([states[start], start, i])
            start = i
    return segs

def min_duration(states, min_len):
    states = list(states)
    while True:
        segs = segments(states)
        short = [k for k, (_, a, b) in enumerate(segs) if b - a < min_len]
        if len(segs) <= 1 or not short:
            return states
        k = short[0]
        neighbors = [segs[j] for j in (k - 1, k + 1) if 0 <= j < len(segs)]
        target = max(neighbors, key=lambda x: x[2] - x[1])[0]
        _, a, b = segs[k]
        states[a:b] = [target] * (b - a)

dt = df["t"].diff().median()
WINDOW = max(1, round(1.5 / dt)) | 1
MIN_LEN = max(1, int(np.ceil(1.5 / dt)))

df["smoothed"] = min_duration(majority_filter(list(df["raw_state"]), WINDOW), MIN_LEN)
plot_states({"raw": df["raw_state"], "smoothed": df["smoothed"]}, "Raw vs smoothed")

## 6. Timeline, durations, and bed events (from smoothed states)

In [ ]:
duration = n_frames / fps
times = list(df["t"])

def mmss(sec):
    return f"{int(sec) // 60:02d}:{int(sec) % 60:02d}"

def seg_end(b):
    return times[b] if b < len(times) else duration

def build_timeline(states):
    return [{"state": st, "start": 0.0 if a == 0 else times[a], "end": seg_end(b),
             "duration": seg_end(b) - (0.0 if a == 0 else times[a])} for st, a, b in segments(states)]

def bed_status(states):
    status, last = [], None
    for x in states:
        if x in IN_STATES:
            last = "in"
        elif x in OUT_STATES:
            last = "out"
        status.append(last)
    first = next((x for x in status if x), "in")
    return [x or first for x in status]

EXIT_CONFIRM, RETURN_CONFIRM = 3.0, 3.0

def detect_events(states, times):
    status = bed_status(states)
    confirmed = status[0]
    events, cancelled, pending = [], [], None
    for i in range(1, len(states)):
        if pending and status[i] == confirmed:
            cancelled.append({**pending, "cancelled_at": times[i]})
            pending = None
        if pending is None and status[i] != confirmed and status[i] != status[i - 1]:
            kind = "bed_exit" if status[i] == "out" else "return_to_bed"
            pending = {"event": kind, "start": times[i], "previous_state": states[i - 1]}
        if pending:
            need = EXIT_CONFIRM if pending["event"] == "bed_exit" else RETURN_CONFIRM
            if times[i] - pending["start"] >= need:
                events.append({**pending, "confirmed": times[i], "current_state": states[i]})
                confirmed = status[i]
                pending = None
    return events, cancelled

smooth_timeline = build_timeline(list(df["smoothed"]))
for seg in smooth_timeline:
    print(f"{mmss(seg['start'])} - {mmss(seg['end'])}  {seg['state']}")

dur = pd.DataFrame(smooth_timeline).groupby("state")["duration"].sum().sort_values(ascending=False)
print("\n", dur.round(1))
print(f"sum of durations: {dur.sum():.1f}s | video duration: {duration:.1f}s")

In [ ]:
smooth_events, smooth_cancelled = detect_events(list(df["smoothed"]), times)
df["bed_status"] = bed_status(list(df["smoothed"]))

for e in smooth_events:
    print(f"{e['event']:14s} start {mmss(e['start'])}  confirmed {mmss(e['confirmed'])}  "
          f"{e['previous_state']} -> {e['current_state']}")
for c in smooth_cancelled:
    print(f"cancelled {c['event']} at {mmss(c['start'])} (back by {mmss(c['cancelled_at'])})")

fig, ax = plt.subplots(figsize=(14, 3))
ax.step(df["t"], (df["bed_status"] == "in").astype(int), where="post", color="black")
for e in smooth_events:
    color = "red" if e["event"] == "bed_exit" else "green"
    ax.axvspan(e["start"], e["confirmed"], color=color, alpha=0.3)
    ax.axvline(e["confirmed"], color=color, ls="--")
    ax.text(e["confirmed"], 1.05, e["event"], color=color, fontsize=9)
for c in smooth_cancelled:
    ax.axvline(c["start"], color="orange", ls=":")
ax.set_yticks([0, 1], ["out", "in"])
ax.set_xlabel("time (s)")
ax.set_title("Bed status | shaded = start→confirmed | orange dotted = cancelled")
plt.tight_layout()
plt.show()

## 7. Ground truth

Placeholder labels. Replace `GT` with the real segments for this video before trusting any metric.

In [ ]:
GT = [
    (0, 18, "LYING_IN_BED"),
    (18, 24, "SITTING_ON_BED"),
    (24, 27, "STANDING"),
]

def gt_at(t):
    for a, b, st in GT:
        if a <= t < b:
            return st
    return None

def durations_from(segs_list):
    out = {}
    for a, b, st in segs_list:
        out[st] = out.get(st, 0.0) + (min(b, duration) - a)
    return out

df["gt"] = df["t"].map(gt_at)
print(df["gt"].isna().sum(), "samples without a label")

## 8. VLM (Amazon Bedrock, OpenAI-compatible API)

The last line lists the models your key can use. Set a vision-capable one as `VLM_MODEL` in `.env`.

In [ ]:
import base64, re, time
from openai import OpenAI, RateLimitError
from pydantic import BaseModel, ValidationError

client = OpenAI(
    api_key=os.environ["AWS_BEARER_TOKEN_BEDROCK"],
    base_url=os.environ["OPENAI_BASE_URL"],
    project=os.getenv("OPENAI_PROJECT_ID") or None,
)

class VLMAnswer(BaseModel):
    state: str
    location: str
    confidence: float
    reasoning: str

PROMPT = """You are analyzing consecutive frames from an indoor camera monitoring an elderly person near a bed.
Answer using only what is visible. If the evidence is insufficient, use UNKNOWN.
Allowed states: LYING_IN_BED, SITTING_ON_BED, SITTING_OUTSIDE_BED, STANDING, WALKING, OUT_OF_BED, LYING_OUTSIDE_BED, UNKNOWN
Allowed locations: bed, floor, chair, standing_area, not_visible, unknown
Question: {question}

Respond with ONLY a JSON object with keys:
"state" (one allowed state), "location" (one allowed location), "confidence" (0 to 1), "reasoning" (one sentence)."""

def frame_at(t):
    return min(samples, key=lambda x: abs(x["t"] - t))["image"]

def small(img, max_side=512):
    h, w = img.shape[:2]
    scale = min(1.0, max_side / max(h, w))
    return cv2.resize(img, (int(w * scale), int(h * scale)))

def to_data_url(img):
    return "data:image/jpeg;base64," + base64.b64encode(cv2.imencode(".jpg", small(img))[1].tobytes()).decode()

def parse_answer(text):
    match = re.search(r"\{.*\}", text or "", re.DOTALL)
    if not match:
        return None
    try:
        return VLMAnswer.model_validate_json(match.group(0))
    except ValidationError:
        return None

def ask_vlm(t, question, model_id=VLM_MODEL, retries=3):
    content = [{"type": "image_url", "image_url": {"url": to_data_url(frame_at(x))}} for x in (t - 1, t, t + 1)]
    content.append({"type": "text", "text": PROMPT.format(question=question)})
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": content}],
                temperature=0,
                max_tokens=400,
            )
            return parse_answer(response.choices[0].message.content)
        except RateLimitError:
            if attempt < retries - 1:
                time.sleep(10 * (attempt + 1))
                continue
            print("VLM call failed: rate limited")
            return None
        except Exception as exc:
            print(f"VLM call failed: {exc}")
            return None

for m in client.models.list():
    print(m.id)

In [ ]:
def check_vlm(t, question="What state is the person in? Are they on the bed, floor, or elsewhere?"):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, x in zip(axes, (t - 1, t, t + 1)):
        ax.imshow(cv2.cvtColor(small(frame_at(x)), cv2.COLOR_BGR2RGB))
        ax.set_title(f"t={x:.1f}s")
        ax.axis("off")
    plt.show()
    start = time.time()
    ans = ask_vlm(t, question)
    print(f"latency: {time.time() - start:.1f}s")
    print(ans)
    return ans

check_vlm(samples[len(samples) // 2]["t"])

## 9. Agent: temporal context first, VLM for what remains

In [ ]:
CONTEXT = max(1, round(8 / dt))
FILL_MAX_SEC = 30
MIN_CONF = 0.6

smoothed = list(df["smoothed"])
trace = []

def log(t, observation, thought, action, finding, conclusion):
    trace.append({"time": mmss(t), "observation": observation, "thought": thought,
                  "action": action, "finding": finding, "conclusion": conclusion})

def look_back(states, idx):
    known = [x for x in states[max(0, idx - CONTEXT): idx] if x != "UNKNOWN"]
    return known[-1] if known else None

def look_ahead(states, idx):
    known = [x for x in states[idx: idx + CONTEXT] if x != "UNKNOWN"]
    return known[0] if known else None

for st, a, b in segments(smoothed):
    if st in ("UNKNOWN", "LYING_OUTSIDE_BED"):
        print(f"flagged: {st:18s} {mmss(times[a])} - {mmss(seg_end(b))}  ({b - a} frames)")

In [ ]:
agent_states = list(smoothed)

for st, a, b in segments(agent_states):
    if st != "UNKNOWN":
        continue
    t0, t1 = times[a], seg_end(b)
    before, after = look_back(agent_states, a), look_ahead(agent_states, b)
    if before is not None and before == after and t1 - t0 <= FILL_MAX_SEC:
        agent_states[a:b] = [before] * (b - a)
        log(t0, f"UNKNOWN for {t1 - t0:.1f}s", "Current frames insufficient; check surrounding context.",
            "look_back + look_ahead", f"{before} before and after", f"Relabel as {before}")
    else:
        log(t0, f"UNKNOWN for {t1 - t0:.1f}s", "Current frames insufficient; check surrounding context.",
            "look_back + look_ahead", f"before={before}, after={after}", "Unresolved, escalate to VLM")

print(pd.Series(agent_states).value_counts())

In [ ]:
for st, a, b in segments(agent_states):
    if st not in ("UNKNOWN", "LYING_OUTSIDE_BED"):
        continue
    t0, t1 = times[a], seg_end(b)
    mid = (t0 + t1) / 2

    if st == "UNKNOWN":
        ans = ask_vlm(mid, "What state is the person in? Are they on the bed, floor, or elsewhere?")
        if ans and ans.state in STATE_ORDER and ans.state != "UNKNOWN" and ans.confidence >= MIN_CONF:
            agent_states[a:b] = [ans.state] * (b - a)
            log(t0, "UNKNOWN after context check", "Ask the VLM to inspect the frames.", "ask_vlm",
                f"{ans.state} on {ans.location} ({ans.confidence:.2f}): {ans.reasoning}", f"Relabel as {ans.state}")
        else:
            log(t0, "UNKNOWN after context check", "Ask the VLM to inspect the frames.", "ask_vlm",
                "VLM unavailable or not confident", "Keep UNKNOWN")
    else:
        ans = ask_vlm(mid, "Is the person lying on the bed or on the floor?")
        if ans and ans.location == "bed" and ans.confidence >= MIN_CONF:
            agent_states[a:b] = ["LYING_IN_BED"] * (b - a)
            log(t0, "Lying outside bed region", "Determine bed or floor.", "ask_vlm",
                f"On bed ({ans.confidence:.2f}): {ans.reasoning}", "Bed region too small; relabel LYING_IN_BED")
        elif ans and ans.location == "floor":
            log(t0, "Lying outside bed region", "Determine bed or floor.", "ask_vlm",
                f"On floor ({ans.confidence:.2f}): {ans.reasoning}", "Possible fall confirmed, ALERT")
        else:
            log(t0, "Lying outside bed region", "Determine bed or floor.", "ask_vlm",
                "VLM unavailable or inconclusive", "Keep as possible fall, ALERT rule applies")

df["agent_state"] = agent_states

for step_ in trace:
    print(f"[{step_['time']}] {step_['observation']}")
    print(f"   thought:    {step_['thought']}")
    print(f"   action:     {step_['action']}")
    print(f"   finding:    {step_['finding']}")
    print(f"   conclusion: {step_['conclusion']}\n")

## 10. State machine

In [ ]:
ALLOWED = {
    "LYING_IN_BED": {"SITTING_ON_BED", "LYING_OUTSIDE_BED"},
    "SITTING_ON_BED": {"LYING_IN_BED", "STANDING", "LYING_OUTSIDE_BED"},
    "STANDING": {"SITTING_ON_BED", "WALKING", "SITTING_OUTSIDE_BED", "LYING_OUTSIDE_BED", "OUT_OF_BED"},
    "WALKING": {"STANDING", "SITTING_ON_BED", "SITTING_OUTSIDE_BED", "LYING_OUTSIDE_BED", "OUT_OF_BED"},
    "SITTING_OUTSIDE_BED": {"STANDING", "LYING_OUTSIDE_BED"},
    "LYING_OUTSIDE_BED": {"SITTING_OUTSIDE_BED", "STANDING", "SITTING_ON_BED"},
    "OUT_OF_BED": {"STANDING", "WALKING"},
}
HOLD = max(1, round(1.0 / dt))

def is_allowed(a, b):
    return a == b or "UNKNOWN" in (a, b) or b in ALLOWED.get(a, set())

def enforce_transitions(states):
    out, flags = [], []
    current, pending, count = None, None, 0
    for i, x in enumerate(states):
        if current is None or x == current:
            current = current or x
            pending, count = None, 0
        elif is_allowed(current, x):
            current, pending, count = x, None, 0
        else:
            count = count + 1 if x == pending else 1
            pending = x
            if count > HOLD:
                start = i - HOLD
                flags.append(f"{mmss(times[start])}: {current} -> {x} (illegal, accepted after persisting)")
                out[start:i] = [x] * HOLD
                current, pending, count = x, None, 0
        out.append(current)
    return out, flags

final, flags = enforce_transitions(list(df["agent_state"]))
df["final"] = final
print("\n".join(flags) or "No illegal transitions")
plot_states({"raw": df["raw_state"], "smoothed": df["smoothed"], "agent": df["agent_state"], "final": df["final"]},
            "Each stage of the pipeline")

## 11. Final timeline, events, and alerts

In [ ]:
states = list(df["final"])
timeline = build_timeline(states)
events, cancelled = detect_events(states, times)
status = bed_status(states)
out_periods = [(times[a], seg_end(b)) for st, a, b in segments(status) if st == "out"]

for seg in timeline:
    print(f"{mmss(seg['start'])} - {mmss(seg['end'])}  {seg['state']}")

In [ ]:
SIT_EDGE_MONITOR = 120
UNKNOWN_MONITOR = 30
OUT_OF_VIEW_MONITOR = 60
LYING_OUTSIDE_ALERT = 5
OUT_OF_BED_ALERT = 600
LEVEL = {"NORMAL": 0, "MONITOR": 1, "ALERT": 2}

alerts = []
for seg in timeline:
    d, t0 = seg["duration"], seg["start"]
    if seg["state"] == "SITTING_ON_BED" and d >= SIT_EDGE_MONITOR:
        alerts.append(("MONITOR", t0 + SIT_EDGE_MONITOR, f"sitting on bed {d:.0f}s"))
    if seg["state"] == "UNKNOWN" and d >= UNKNOWN_MONITOR:
        alerts.append(("MONITOR", t0 + UNKNOWN_MONITOR, f"state unknown {d:.0f}s"))
    if seg["state"] == "OUT_OF_BED" and d >= OUT_OF_VIEW_MONITOR:
        alerts.append(("MONITOR", t0 + OUT_OF_VIEW_MONITOR, f"out of view {d:.0f}s"))
    if seg["state"] == "LYING_OUTSIDE_BED" and d >= LYING_OUTSIDE_ALERT:
        alerts.append(("ALERT", t0 + LYING_OUTSIDE_ALERT, f"lying outside bed {d:.0f}s, possible fall"))

for a, b in out_periods:
    if b - a >= OUT_OF_BED_ALERT:
        alerts.append(("ALERT", a + OUT_OF_BED_ALERT, f"out of bed {b - a:.0f}s"))

for e in events:
    e["decision"] = "MONITOR" if e["event"] == "bed_exit" else "NORMAL"

overall = max([a[0] for a in alerts] + [e["decision"] for e in events], key=LEVEL.get, default="NORMAL")

for level, t, reason in sorted(alerts, key=lambda x: x[1]):
    print(f"{level:8s} {mmss(t)}  {reason}")
print("\nOverall decision:", overall)

In [ ]:
import json

act = pd.DataFrame(timeline).groupby("state")["duration"].sum()
in_bed_sec = sum(seg_end(b) - times[a] for st, a, b in segments(status) if st == "in")

summary = {
    "observation_duration_sec": round(duration, 1),
    "activity_duration_sec": {k.lower(): round(v, 1) for k, v in act.items()},
    "bed_exit_count": sum(e["event"] == "bed_exit" for e in events),
    "bed_return_count": sum(e["event"] == "return_to_bed" for e in events),
    "total_in_bed_sec": round(in_bed_sec, 1),
    "total_out_of_bed_sec": round(duration - in_bed_sec, 1),
    "longest_out_of_bed_period_sec": round(max([b - a for a, b in out_periods], default=0), 1),
    "final_state": states[-1].lower(),
    "overall_decision": overall,
}

event_out = [{"event": e["event"], "start_time": mmss(e["start"]), "confirmed_time": mmss(e["confirmed"]),
              "previous_state": e["previous_state"].lower(), "current_state": e["current_state"].lower(),
              "decision": e["decision"]} for e in events]

print(json.dumps(summary, indent=2))
print(json.dumps(event_out, indent=2))

## 12. Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

ev = df.dropna(subset=["gt"])
stages = {"raw rules": "raw_state", "+ smoothing": "smoothed", "+ agent (VLM)": "agent_state", "+ state machine": "final"}

ablation = pd.DataFrame([
    {"stage": name, "accuracy": round(accuracy_score(ev["gt"], ev[col]), 3),
     "unknown_frames": int((ev[col] == "UNKNOWN").sum())}
    for name, col in stages.items()
])
ablation

In [ ]:
labels = [x for x in STATE_ORDER if x in set(ev["gt"]) | set(ev["final"])]
print(classification_report(ev["gt"], ev["final"], labels=labels, zero_division=0))
ConfusionMatrixDisplay.from_predictions(ev["gt"], ev["final"], labels=labels, xticks_rotation=45)
plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png", dpi=120)
plt.show()

pred_dur = pd.DataFrame(timeline).groupby("state")["duration"].sum()
dur_table = pd.DataFrame({"ground_truth": durations_from(GT), "predicted": pred_dur}).fillna(0)
dur_table["error_sec"] = (dur_table["ground_truth"] - dur_table["predicted"]).abs()
print(dur_table.round(1))

gt_events, _ = detect_events([gt_at(t) or "UNKNOWN" for t in times], times)
TOL = 5.0
used, tp = set(), 0
for g in gt_events:
    for k, p in enumerate(events):
        if k not in used and p["event"] == g["event"] and abs(p["start"] - g["start"]) <= TOL:
            used.add(k)
            tp += 1
            break
false_events = [mmss(p["start"]) + " " + p["event"] for k, p in enumerate(events) if k not in used]
print(f"\nevents: gt={len(gt_events)} predicted={len(events)} matched={tp} false={false_events}")
print(f"precision: {tp / len(events):.2f}" if events else "precision: n/a (no predictions)")
print(f"recall:    {tp / len(gt_events):.2f}" if gt_events else "recall: n/a (no ground-truth events)")

In [ ]:
wrong = [f != g for f, g in zip(df["final"], df["gt"])]
error_segs = [(a, b) for flag, a, b in segments(wrong) if flag and b - a >= 2]
print(f"{len(error_segs)} error segments")

if error_segs:
    shown = error_segs[:6]
    fig, axes = plt.subplots(len(shown), 1, figsize=(10, 5 * len(shown)), squeeze=False)
    for ax, (a, b) in zip(axes[:, 0], shown):
        mid = (a + b) // 2
        row = df.iloc[mid]
        ax.imshow(cv2.cvtColor(samples[mid]["result"].plot(), cv2.COLOR_BGR2RGB))
        ax.set_title(f"{mmss(times[a])}-{mmss(times[b - 1])} | truth: {row['gt']} | predicted: {row['final']}\n"
                     f"torso={row['torso_angle']:.0f}° knee={row['knee_angle']:.0f}° "
                     f"hips_in_bed={row['hip_in_bed']} visible_kp={row['visible_kp']:.0f}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 13. Save outputs

In [ ]:
(OUT / "timeline.txt").write_text(
    "\n".join(f"{mmss(x['start'])} - {mmss(x['end'])}  {x['state']}" for x in timeline) + "\n")
(OUT / "summary.json").write_text(json.dumps(summary, indent=2))
(OUT / "events.json").write_text(json.dumps(event_out, indent=2))
(OUT / "alerts.json").write_text(json.dumps(
    {"overall_decision": overall, "alerts": [{"level": l, "time": mmss(t), "reason": r} for l, t, r in alerts]}, indent=2))
(OUT / "agent_trace.json").write_text(json.dumps(trace, indent=2))
ablation.to_csv(OUT / "ablation.csv", index=False)
dur_table.to_csv(OUT / "duration_error.csv")
df.to_csv(OUT / "frames.csv", index=False)

print("saved:", sorted(p.name for p in OUT.iterdir()))

In [ ]:
LEVEL_COLOR = {"NORMAL": (0, 200, 0), "MONITOR": (0, 200, 255), "ALERT": (0, 0, 255)}

def level_at(t):
    level = "NORMAL"
    for l, at, _ in alerts:
        if at <= t and LEVEL[l] > LEVEL[level]:
            level = l
    return level

h, w = samples[0]["image"].shape[:2]
writer = cv2.VideoWriter(str(OUT / "annotated.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), SAMPLE_FPS, (w, h))
for s_, st in zip(samples, df["final"]):
    img = s_["result"].plot()
    cv2.polylines(img, [bed_poly], True, (255, 0, 0), 2)
    level = level_at(s_["t"])
    cv2.rectangle(img, (0, 0), (w, 34), (0, 0, 0), -1)
    cv2.putText(img, f"{mmss(s_['t'])}  {st}  [{level}]", (10, 24),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, LEVEL_COLOR[level], 2)
    writer.write(img)
writer.release()
print("saved", OUT / "annotated.mp4")